# Customer Churn Modeling

Se nos suministra un conjunto de datos cualitativos y cuantitativos de clientes de una empresa de telecomunicaciones india. El objetivo es es encontrar las acciones concretas que nos ayuden a prevenir que un cliente haga churn (abandone la empres). Para ello, vamos a utilizar las técnicas estadísticas y de modelado de datos hasta ahora en la asignatura.

## Load Data and Libraries

Aquí se cargan las librerías necesarias y el conjunto de datos.

In [1]:
import pandas as pd
import numpy as np

# Libraries of visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set options
pd.set_option('display.max_columns', None)
sns.set_style('darkgrid')

# Seed used for reproducibility
SEED = 2026

In [2]:
# Load data in csv format
data = pd.read_csv('../data/customer_churn_data.csv')

## 1. Analisis exploratorio de los datos

En esta sección vamos a relaizar un análisis exploratorio de los datos para entender mejor como están compuestas las vairbles. Además realizaremos un análisis de estadística descriptiva y de calidad general de los datos. Crearemos visualizaciones de datos para entender mejor las relaciones entre las variables y entender su distribución. Preliminarmente, se nos ha suministrado una descripción de cada una de las variables, que se muestran a continuación:

* `device` user’s – device brand (Categorical)
* `first_payment_amount`  – user’s first payment amount(Numeric)
* `age`  – user’s age(Numeric or categorical?)
* `city`  – user’s city(Categorical)
* `number_of_cards`  – #of cards user owns
* `payments_initiated`  – #of payments initiated by user
* `payments_failed`  – #of payments failed
* `payments_completed` – #of payments completed
* `payments_completed_amount_first_7days`  – amt of payment completed in first 7 days of joining
* `reward_purchase_count_first_7days` – #of rewards claimed in first 7 days
* `coins_redeemed_first_7days` – coins redeemed in first 7 days
* `is_referral` – is user a referred user
* `visits_feature_1`  – #of visits made by user to product feature 1
* `visits_feature_2` – #of visits made by user to product feature 2
* `given_permission_1` – has user given permission 1
* `given_permission_2` – has user given permission 2
* `user_id` – user identifier
* `is_churned` – whether user churned

`is_churned` es la variable objetivo, que indica si el cliente ha abandonado o no la empresa. El resto de las variables son carácteristicas de los clientes.

Ya tenemos cargado los datos, veamos que dimensiones tienen y que tipo de variables tenemos en el momento de la carga de los datos.

### 1.1 Shape Data y Análisis descriptivo

Exploremos incialmente los datos para entender su forma y tipo de variables, además de su distribucion y calidad general.

In [3]:
# Shape of data
print(f'Shape of data (rows, columns): {data.shape}\n')
# Data types of columns
print(f'Data types of columns:\n{data.info()}\n')

Shape of data (rows, columns): (104143, 18)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104143 entries, 0 to 104142
Data columns (total 18 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   device                                 104025 non-null  object 
 1   first_payment_amount                   104143 non-null  int64  
 2   age                                    104001 non-null  float64
 3   city                                   98301 non-null   object 
 4   number_of_cards                        103671 non-null  float64
 5   payments_initiated                     103671 non-null  float64
 6   payments_failed                        103671 non-null  float64
 7   payments_completed                     103671 non-null  float64
 8   payments_completed_amount_first_7days  103671 non-null  float64
 9   reward_purchase_count_first_7days      80879 non-null   float64
 10  coins_redee

In [4]:
# Summary of missing values
missing_values = data.isnull().sum()
missing_values = np.round(missing_values /  len(data) * 100, 2)
missing_values = missing_values[missing_values > 0]
missing_values.columns = ['Missing Values (%)']
# Convert to string with percentage format
missing_values = missing_values.apply(lambda x: f'{x}%')
print(f'Missing values in each column:\n{missing_values}\n')

Missing values in each column:
device                                    0.11%
age                                       0.14%
city                                      5.61%
number_of_cards                           0.45%
payments_initiated                        0.45%
payments_failed                           0.45%
payments_completed                        0.45%
payments_completed_amount_first_7days     0.45%
reward_purchase_count_first_7days        22.34%
coins_redeemed_first_7days                0.45%
visits_feature_1                          2.54%
visits_feature_2                          2.54%
dtype: object



La data esta compuesta por 104.142 filas con 18 columnas que describen las variables. Tenemos una mezcla de variabels numéricas y categóricas. En cuanto la calidad de los datos, encontramos valores nulos no especialmente preocupantes en algunas variables con menos del 0.5% de los datos faltantes, pero otras como `city` con 5.61%, `visits_feature_1` con 2.54%, `visits_feature_2` con 2.54% y `reward_pruchase_count_first_7days` con 22.34% siendo el mas crítico de todos. Podría ser necesario tratar con ellos antes de proceder con el análisis y modelado.

Veamos ahora como estan las variables continuas en una exploración rapida de su estadística descriptiva.

In [ ]:
# Descriptive statistics of numerical variables
data.describe()

,first_payment_amount,age,number_of_cards,payments_initiated,payments_failed,payments_completed,payments_completed_amount_first_7days,reward_purchase_count_first_7days,coins_redeemed_first_7days,visits_feature_1,visits_feature_2,given_permission_1,given_permission_2,user_id,is_churned
count,104143.000000,104001.000000,103671.000000,103671.000000,103671.000000,103671.000000,103671.000000,80879.000000,103671.000000,101497.000000,101497.000000,104143.000000,104143.000000,104143.000000,104143.000000
mean,34.771353,32.688291,1.989148,2.847402,0.439940,1.830676,55.965381,2.042075,22.494102,0.239643,0.126861,0.852318,0.723332,251829.347273,0.286808
std,73.181032,7.821752,1.871436,4.223077,1.061101,3.210357,130.720128,3.731290,64.122134,0.624616,0.445573,0.354786,0.447353,146009.671697,0.452273
min,0.000000,20.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,27.000000,1.000000,1.000000,0.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,124478.500000,0.000000
50%,12.000000,31.000000,1.000000,2.000000,0.000000,1.000000,21.000000,1.000000,3.000000,0.000000,0.000000,1.000000,1.000000,252377.000000,0.000000
75%,37.000000,36.000000,3.000000,3.000000,0.000000,2.000000,64.000000,3.000000,20.000000,0.000000,0.000000,1.000000,1.000000,378186.500000,1.000000
max,4370.000000,80.000000,30.000000,359.000000,30.000000,337.000000,11107.000000,304.000000,8857.000000,17.000000,23.000000,1.000000,1.000000,503533.000000,1.000000


Existen variables `int64` que realmente se tratan de variables categóricas de 1 o 0. Estas variables son `given_permission_1`, `given_permission_2`, otra que debemos eliminar es `user_id` ya que se trata unicamente de un identificador único para cada usuario. Por tanto, las trateremos como variables categóricas en la fase exploratoria.

Primero hagamos una visuazlización de las distribuciones de las variables numéricas para entender mejor su comportamiento. Para ello, podemos utilizar histogramas o violinesplot, esto nos ayudará a entender la distribución de cada variable y detectar posibles outliers o asimetrías en los datos.

In [18]:
# data_numeric = data.drop(['given_permission_1', 'given_permission_2', 'user_id'], axis=1).select_dtypes(include=['int64', 'float64']).copy()

# # Create Canva with matplotlib and seaborn
# fig, ax = plt.subplots(2, 12, figsize=(32, 12), sharey = True)

# # Loop through each numeric column and create a histogram
# for i, column in enumerate(data_numeric.columns):
#     # Hist Plot with KDE
#     sns.histplot(data_numeric[column], kde=True, ax=ax[0, i])
#     ax[0, i].set_title(f'{column}', fontsize = 10)
#     ax[0, i].set_xlabel("")
#     ax[0, i].set_ylabel('Frequency')

#     # Violin Plot
#     sns.violinplot(data_numeric[column], orient='v', ax=ax[1, i])
#     ax[1, i].set_title(f'{column}', fontsize = 10)
    

# plt.show()

In [7]:
data_numeric.shape

(104143, 13)